In [26]:
# Welcome to your new notebook
# Type here in the cell editor to add code!
%pip install -U semantic-link semantic-link-sempy requests 

StatementMeta(, 264a27ac-e05e-4418-ba01-f41928d9e8b8, 53, Submitted, Running, Running, True)


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sempy.fabric as fabric
import requests
from notebookutils import credentials
import pandas as pd
import time



StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
df_capacity = fabric.list_capacities()
print(df_capacity)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
#for index, row in df_capacity.iterrows():
#    print(row["Id"])
#    df_ws1 = fabric.list_workspaces()
#    print(df_ws1)


StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
df_ws = fabric.list_workspaces()
df_ws

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
workspace_id= fabric.get_workspace_id()
print(workspace_id)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
items = fabric.list_items(workspace=workspace_id)

display(items)

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
# get token
token = credentials.getToken(
    "https://analysis.windows.net/powerbi/api"
)

print(token[:100])

# add token to the header
headers = {
    "Authorization": f"Bearer {token}",
    "Content-Type": "application/json"
}

StatementMeta(, , -1, Waiting, , Waiting, True)

**Charge Back**

In [ ]:

#report_id = "caf377df-a7d0-4a0a-91af-e3909972db3d"

item_name = "rpt_chargeback"

report_id = items.loc[
    items["Display Name"] == item_name,
    "Id"
].iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)

print(url)

#create CSV

payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Get the export id

export_id = result["id"]

print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"
while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()

    print(result["status"])

    if result["status"] == "Succeeded":
        break

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"

print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

# Save to Lakehouse Files area

output_file = "chargeback.csv"

with open(output_file, "wb") as f:
    f.write(r.content)

print("File saved:", output_file)



# create a dataframe

df = pd.read_csv(
    "chargeback.csv"
)

display(df.head())

# create delta table
spark_df = spark.createDataFrame(df)

spark_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("chargeback")

print("table chargeback created")
print("chargeback load completed...") 






StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
spark_df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(
      "abfss://DemoWS@onelake.dfs.fabric.microsoft.com/chargeback.Lakehouse/Tables/dbo/chargeback"
  )

StatementMeta(, , -1, Waiting, , Waiting, True)

**Workspaces**

In [ ]:

#report_id = "88b76547-ca5a-40ea-815b-e7fb6cf93bb5"

item_name = "rpt_workspaces"

report_id = items.loc[
    items["Display Name"] == item_name,
    "Id"
].iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)

print(url)

#create CSV

payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Get the export id

export_id = result["id"]

print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"

while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()
    

    print(result["status"])

    if result["status"] == "Succeeded":
        break

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"

print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

# Save to Lakehouse Files area

output_file = "workspaces.csv"

with open(output_file, "wb") as f:
    f.write(r.content)

print("File saved:", output_file)

# create a dataframe

df = pd.read_csv(
    "workspaces.csv"
)

display(df.head())

# create delta table
spark_df = spark.createDataFrame(df)

spark_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("workspaces")

print("table workspaces created")
print("workspaces load completed...")    

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
spark_df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(
      "abfss://DemoWS@onelake.dfs.fabric.microsoft.com/chargeback.Lakehouse/Tables/dbo/workspaces"
  )

StatementMeta(, , -1, Waiting, , Waiting, True)

**Items**

In [ ]:

#report_id = "d4184a3c-9832-42d0-9d0d-1367d9d90959"

item_name = "rpt_items"

report_id = items.loc[
    items["Display Name"] == item_name,
    "Id"
].iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)

print(url)

#create CSV

payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Get the export id

export_id = result["id"]

print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"
while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()

    print(result["status"])

    if result["status"] == "Succeeded":
        break

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"

print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

# Save to Lakehouse Files area

output_file = "items.csv"

with open(output_file, "wb") as f:
    f.write(r.content)

print("File saved:", output_file)

# create a dataframe

df = pd.read_csv(
    "items.csv"
)

display(df.head())

# create delta table
spark_df = spark.createDataFrame(df)

spark_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("items")

print("table items created")
print("items load completed...")    

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
spark_df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(
      "abfss://DemoWS@onelake.dfs.fabric.microsoft.com/chargeback.Lakehouse/Tables/dbo/items"
  )

StatementMeta(, , -1, Waiting, , Waiting, True)

**Dates**

In [ ]:

#report_id = "0de808b3-f9ae-4067-8195-77176b27b81c"

item_name = "rpt_dates"

report_id = items.loc[
    items["Display Name"] == item_name,
    "Id"
].iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)

print(url)

#create CSV

payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Get the export id

export_id = result["id"]

print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"
while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()

    print(result["status"])

    if result["status"] == "Succeeded":
        break

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"

print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

# Save to Lakehouse Files area

output_file = "dates.csv"

with open(output_file, "wb") as f:
    f.write(r.content)

print("File saved:", output_file)

# create a dataframe

df = pd.read_csv(
    "dates.csv"
)

display(df.head())

# create delta table
spark_df = spark.createDataFrame(df)

spark_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("dates")

print("table dates created")
print("dates load completed...")    

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:
spark_df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(
      "abfss://DemoWS@onelake.dfs.fabric.microsoft.com/chargeback.Lakehouse/Tables/dbo/dates"
  )

StatementMeta(, , -1, Waiting, , Waiting, True)

**capacities**

In [ ]:

#report_id = "da196fd2-50e6-4eed-a1b9-0dc8bb6015af"

item_name = "rpt_capacities"

report_id = items.loc[
    items["Display Name"] == item_name,
    "Id"
].iloc[0]

# build report url
url = (
    f"https://api.powerbi.com/v1.0/myorg"
    f"/reports/{report_id}"
    f"/ExportTo"
)

print(url)

#create CSV

payload = {
    "format": "CSV"
}

headers = {
    "Authorization": f"Bearer {token}"
}

response = requests.post(
    url,
    headers=headers,
    json=payload
)
result = response.json()
print(result)

# Get the export id

export_id = result["id"]

print(export_id)

# wait for export to be completed.
status_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}"
while True:

    r = requests.get(
        status_url,
        headers=headers
    )

    result = r.json()

    print(result["status"])

    if result["status"] == "Succeeded":
        break

    time.sleep(10)


# download the CSV

# Resource location from your export result
download_url = f"https://api.powerbi.com/v1.0/myorg/reports/{report_id}/exports/{export_id}/file"

print(download_url)

r = requests.get(
    download_url,
    headers=headers
)
print("download status:")
print(r.status_code)

# Save to Lakehouse Files area

output_file = "capacities.csv"

with open(output_file, "wb") as f:
    f.write(r.content)

print("File saved:", output_file)

# create a dataframe

df = pd.read_csv(
    "capacities.csv"
)

display(df.head())

# create delta table
spark_df = spark.createDataFrame(df)

spark_df.write \
    .mode("overwrite") \
    .format("delta") \
    .saveAsTable("capacities")

print("table capacities created")
print("capacities load completed...")    

StatementMeta(, , -1, Waiting, , Waiting, True)

In [ ]:


spark_df.write \
  .format("delta") \
  .mode("overwrite") \
  .save(
      "abfss://DemoWS@onelake.dfs.fabric.microsoft.com/chargeback.Lakehouse/Tables/dbo/capacities"
  )

StatementMeta(, , -1, Waiting, , Waiting, True)